In [0]:
%python
# Cell 2: Updated configurations for Unity Catalog Volumes
from pyspark.sql.functions import current_timestamp

# Absolute paths to your files inside the Unity Catalog Volume
FILE_PATH_POLICIES = "/Volumes/workspace/default/first_db_project/policies_master.csv"
FILE_PATH_CLAIMS   = "/Volumes/workspace/default/first_db_project/claims_transactions.csv"
FILE_PATH_FUNNEL   = "/Volumes/workspace/default/first_db_project/web_funnel_events.json"

In [0]:
%python
# Ingest Structured Data from the Volume

# 1. Read Raw Policies CSV
raw_policies_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(FILE_PATH_POLICIES)

# Append tracking metadata column
bronze_policies_df = raw_policies_df.withColumn("ingested_at", current_timestamp())

# 2. Read Raw Claims CSV
raw_claims_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(FILE_PATH_CLAIMS)

# Append tracking metadata column
bronze_claims_df = raw_claims_df.withColumn("ingested_at", current_timestamp())

# Show a quick preview to verify it reads perfectly!
display(bronze_policies_df.limit(5))

policy_id,user_id,policy_type,premium_amount,signup_date,region,ingested_at
POL_5001,USR_1001,Health,778.33,2026-02-01,East,2026-07-01T21:15:18.622Z
POL_5002,USR_1002,Auto,848.84,2026-03-28,North,2026-07-01T21:15:18.622Z
POL_5003,USR_1003,Health,2730.45,2026-03-17,North,2026-07-01T21:15:18.622Z
POL_5004,USR_1004,Home,579.46,2026-01-28,North,2026-07-01T21:15:18.622Z
POL_5005,USR_1005,Auto,1763.39,2026-03-13,North,2026-07-01T21:15:18.622Z


In [0]:
%python
# Cell 3: Ingest Semi-Structured Web Events into Bronze Layer
# Read Raw Funnel JSON
raw_funnel_df = spark.read \
    .format("json") \
    .load(FILE_PATH_FUNNEL)

# Append tracking metadata column
bronze_funnel_df = raw_funnel_df.withColumn("ingested_at", current_timestamp())

# Show a quick preview to verify it worked!
display(bronze_funnel_df.limit(5))

event_name,session_id,timestamp,user_id,ingested_at
Quote_Requested,SSN_259784,2026-01-30T00:04:00,USR_1001,2026-07-01T21:15:27.071Z
Document_Uploaded,SSN_259784,2026-01-30T00:07:00,USR_1001,2026-07-01T21:15:27.071Z
Application_Submitted,SSN_259784,2026-01-30T00:12:00,USR_1001,2026-07-01T21:15:27.071Z
Quote_Requested,SSN_587623,2026-03-25T00:14:00,USR_1002,2026-07-01T21:15:27.071Z
Document_Uploaded,SSN_587623,2026-03-25T00:25:00,USR_1002,2026-07-01T21:15:27.071Z


In [0]:
%python
# Cell 4: Create Bronze Tables in Delta Format inside Unity Catalog

# Save Policies
bronze_policies_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_insurance_policies")

# Save Claims
bronze_claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_insurance_claims")

# Save Web Events
bronze_funnel_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_web_events")

print("Success! All 3 Bronze tables have been safely saved to your Delta Lake Catalog.")

Success! All 3 Bronze tables have been safely saved to your Delta Lake Catalog.


In [0]:
SHOW TABLES IN workspace.default;

select * from workspace.default.bronze_insurance_claims limit 10;

claim_id,policy_id,claim_amount,incident_date,claim_status,ingested_at
CLM_8001,POL_5002,463.62,2026-04-02,Denied,2026-07-01T21:15:43.272Z
CLM_8004,POL_5005,3413.32,2026-03-26,Under_Review,2026-07-01T21:15:43.272Z
CLM_8005,POL_5006,1890.6,2026-02-20,Under_Review,2026-07-01T21:15:43.272Z
CLM_8006,POL_5007,543.64,2026-02-03,Approved,2026-07-01T21:15:43.272Z
CLM_8008,POL_5009,317.53,2026-02-07,Under_Review,2026-07-01T21:15:43.272Z
CLM_8010,POL_5011,1069.93,2026-03-21,Approved,2026-07-01T21:15:43.272Z
CLM_8014,POL_5015,108.06,2026-03-01,Denied,2026-07-01T21:15:43.272Z
CLM_8015,POL_5016,2733.34,2026-04-20,Denied,2026-07-01T21:15:43.272Z
CLM_8018,POL_5019,1716.6,2026-02-14,Under_Review,2026-07-01T21:15:43.272Z
CLM_8019,POL_5020,448.61,2026-04-12,Denied,2026-07-01T21:15:43.272Z
